In [1]:
import gc
import math
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")


# ============================================================
# Config
# ============================================================
TRAIN_PATH = "/kaggle/input/competitions/ts-forecasting/train.parquet"
TEST_PATH = "/kaggle/input/competitions/ts-forecasting/test.parquet"
SUBMISSION_PATH = "/kaggle/working/submission.csv"
TARGET_COL = "y_target"   # change if your target column name is different
ID_COL = "id"
TS_COL = "ts_index"
WEIGHT_COL = "weight"
CATEGORICAL_COLS = ["code", "sub_code", "sub_category", "horizon"]
HORIZON_VALUES = [1, 3, 10, 25]
N_FOLDS = 4
USE_RECENCY_WEIGHT = True
RECENCY_HALFLIFE = 800.0
EARLY_STOPPING_ROUNDS = 200
VERBOSE_EVAL = 200
RANDOM_STATE = 42

# Set this to False if the test file does not contain rows ordered in a way that
# makes sequential group updates meaningful for you.
USE_SIMPLE_TEST_HISTORY = True


# ============================================================
# Metric
# ============================================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))


def weighted_rmse_score(y_true: np.ndarray, y_pred: np.ndarray, w: np.ndarray) -> float:
    denom = np.sum(w * (y_true ** 2))
    if denom <= 0:
        return 0.0
    ratio = np.sum(w * ((y_true - y_pred) ** 2)) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))


# ============================================================
# Utilities
# ============================================================
def reduce_memory(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_integer_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        elif pd.api.types.is_float_dtype(col_type):
            df[col] = df[col].astype(np.float32)
    return df


def get_feature_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("feature_")]


def add_rowwise_meta_features(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    X = df[feature_cols]
    df["feat_mean"] = X.mean(axis=1).astype(np.float32)
    df["feat_std"] = X.std(axis=1).astype(np.float32)
    df["feat_min"] = X.min(axis=1).astype(np.float32)
    df["feat_max"] = X.max(axis=1).astype(np.float32)
    df["feat_range"] = (df["feat_max"] - df["feat_min"]).astype(np.float32)
    df["feat_q25"] = X.quantile(0.25, axis=1).astype(np.float32)
    df["feat_q75"] = X.quantile(0.75, axis=1).astype(np.float32)
    df["feat_iqr"] = (df["feat_q75"] - df["feat_q25"]).astype(np.float32)
    df["feat_nan_frac"] = X.isna().mean(axis=1).astype(np.float32)
    df["feat_pos_frac"] = (X > 0).mean(axis=1).astype(np.float32)
    df["feat_neg_frac"] = (X < 0).mean(axis=1).astype(np.float32)
    df["feat_l2"] = np.sqrt((X.fillna(0.0) ** 2).sum(axis=1)).astype(np.float32)
    return df


def add_basic_time_features(df: pd.DataFrame, max_ts: int) -> pd.DataFrame:
    df["ts_norm"] = (df[TS_COL] / max_ts).astype(np.float32)
    df["ts_mod7"] = (df[TS_COL] % 7).astype(np.int16)
    df["ts_mod30"] = (df[TS_COL] % 30).astype(np.int16)
    df["ts_log1p"] = np.log1p(df[TS_COL]).astype(np.float32)
    return df


@dataclass
class GroupFeatureSpec:
    group_cols: List[str]
    prefix: str


GROUP_SPECS = [
    GroupFeatureSpec(["code", "horizon"], "g_code_h"),
    GroupFeatureSpec(["sub_code", "horizon"], "g_subcode_h"),
    GroupFeatureSpec(["sub_category", "horizon"], "g_subcat_h"),
    GroupFeatureSpec(["code", "sub_code", "horizon"], "g_code_subcode_h"),
]


# ============================================================
# Train-side causal history features
# ============================================================
def add_train_history_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["code", "sub_code", "sub_category", "horizon", TS_COL]).copy()

    for spec in GROUP_SPECS:
        group_obj = df.groupby(spec.group_cols, sort=False)[TARGET_COL]

        lag1 = group_obj.shift(1)
        lag2 = group_obj.shift(2)
        lag3 = group_obj.shift(3)
        lag5 = group_obj.shift(5)

        df[f"{spec.prefix}_lag1"] = lag1.astype(np.float32)
        df[f"{spec.prefix}_lag2"] = lag2.astype(np.float32)
        df[f"{spec.prefix}_lag3"] = lag3.astype(np.float32)
        df[f"{spec.prefix}_lag5"] = lag5.astype(np.float32)

        df[f"{spec.prefix}_roll3_mean"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).rolling(3).mean().reset_index(level=spec.group_cols, drop=True).astype(np.float32)
        )
        df[f"{spec.prefix}_roll5_mean"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).rolling(5).mean().reset_index(level=spec.group_cols, drop=True).astype(np.float32)
        )
        df[f"{spec.prefix}_roll10_mean"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).rolling(10).mean().reset_index(level=spec.group_cols, drop=True).astype(np.float32)
        )
        df[f"{spec.prefix}_roll3_std"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).rolling(3).std().reset_index(level=spec.group_cols, drop=True).astype(np.float32)
        )
        df[f"{spec.prefix}_roll5_std"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).rolling(5).std().reset_index(level=spec.group_cols, drop=True).astype(np.float32)
        )
        df[f"{spec.prefix}_exp_mean"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).expanding().mean().reset_index(level=spec.group_cols, drop=True).astype(np.float32)
        )
        df[f"{spec.prefix}_ewm5"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).transform(lambda s: s.ewm(span=5, adjust=False).mean()).astype(np.float32)
        )
        df[f"{spec.prefix}_ewm10"] = (
            lag1.groupby([df[c] for c in spec.group_cols]).transform(lambda s: s.ewm(span=10, adjust=False).mean()).astype(np.float32)
        )
        df[f"{spec.prefix}_hist_count"] = group_obj.cumcount().astype(np.int32)

        df[f"{spec.prefix}_lag1_minus_lag2"] = (df[f"{spec.prefix}_lag1"] - df[f"{spec.prefix}_lag2"]).astype(np.float32)
        df[f"{spec.prefix}_lag1_over_roll5"] = (
            df[f"{spec.prefix}_lag1"] / (df[f"{spec.prefix}_roll5_mean"].abs() + 1e-6)
        ).astype(np.float32)
        df[f"{spec.prefix}_z_vs_roll10"] = (
            (df[f"{spec.prefix}_lag1"] - df[f"{spec.prefix}_roll10_mean"]) /
            (df[f"{spec.prefix}_roll5_std"].abs() + 1e-6)
        ).astype(np.float32)

    return df


# ============================================================
# Test-side approximate history features from train history
# ============================================================
def build_group_history_cache(train_df: pd.DataFrame) -> Dict[str, Dict[Tuple, Dict[str, float]]]:
    cache: Dict[str, Dict[Tuple, Dict[str, float]]] = {}
    for spec in GROUP_SPECS:
        cache[spec.prefix] = {}
        ordered = train_df.sort_values(spec.group_cols + [TS_COL])
        grouped = ordered.groupby(spec.group_cols, sort=False)
        for key, g in grouped:
            y = g[TARGET_COL].astype(np.float32).values
            if len(y) == 0:
                continue
            cache[spec.prefix][key if isinstance(key, tuple) else (key,)] = {
                "lag1": float(y[-1]),
                "lag2": float(y[-2]) if len(y) >= 2 else float(y[-1]),
                "lag3": float(y[-3]) if len(y) >= 3 else float(y[-1]),
                "lag5": float(y[-5]) if len(y) >= 5 else float(y[-1]),
                "roll3_mean": float(np.mean(y[-3:])),
                "roll5_mean": float(np.mean(y[-5:])),
                "roll10_mean": float(np.mean(y[-10:])),
                "roll3_std": float(np.std(y[-3:])) if len(y) >= 2 else 0.0,
                "roll5_std": float(np.std(y[-5:])) if len(y) >= 2 else 0.0,
                "exp_mean": float(np.mean(y)),
                "ewm5": float(pd.Series(y).ewm(span=5, adjust=False).mean().iloc[-1]),
                "ewm10": float(pd.Series(y).ewm(span=10, adjust=False).mean().iloc[-1]),
                "hist_count": float(len(y)),
            }
    return cache


def add_test_history_features(test_df: pd.DataFrame, history_cache: Dict[str, Dict[Tuple, Dict[str, float]]]) -> pd.DataFrame:
    out = test_df.copy()

    for spec in GROUP_SPECS:
        prefix_cache = history_cache[spec.prefix]
        keys = list(zip(*[out[c].tolist() for c in spec.group_cols]))

        def get_stat(key_tuple: Tuple, stat: str) -> float:
            return float(prefix_cache.get(key_tuple, {}).get(stat, np.nan))

        out[f"{spec.prefix}_lag1"] = [get_stat(k, "lag1") for k in keys]
        out[f"{spec.prefix}_lag2"] = [get_stat(k, "lag2") for k in keys]
        out[f"{spec.prefix}_lag3"] = [get_stat(k, "lag3") for k in keys]
        out[f"{spec.prefix}_lag5"] = [get_stat(k, "lag5") for k in keys]
        out[f"{spec.prefix}_roll3_mean"] = [get_stat(k, "roll3_mean") for k in keys]
        out[f"{spec.prefix}_roll5_mean"] = [get_stat(k, "roll5_mean") for k in keys]
        out[f"{spec.prefix}_roll10_mean"] = [get_stat(k, "roll10_mean") for k in keys]
        out[f"{spec.prefix}_roll3_std"] = [get_stat(k, "roll3_std") for k in keys]
        out[f"{spec.prefix}_roll5_std"] = [get_stat(k, "roll5_std") for k in keys]
        out[f"{spec.prefix}_exp_mean"] = [get_stat(k, "exp_mean") for k in keys]
        out[f"{spec.prefix}_ewm5"] = [get_stat(k, "ewm5") for k in keys]
        out[f"{spec.prefix}_ewm10"] = [get_stat(k, "ewm10") for k in keys]
        out[f"{spec.prefix}_hist_count"] = [get_stat(k, "hist_count") for k in keys]

        out[f"{spec.prefix}_lag1_minus_lag2"] = (out[f"{spec.prefix}_lag1"] - out[f"{spec.prefix}_lag2"]).astype(np.float32)
        out[f"{spec.prefix}_lag1_over_roll5"] = (
            out[f"{spec.prefix}_lag1"] / (out[f"{spec.prefix}_roll5_mean"].abs() + 1e-6)
        ).astype(np.float32)
        out[f"{spec.prefix}_z_vs_roll10"] = (
            (out[f"{spec.prefix}_lag1"] - out[f"{spec.prefix}_roll10_mean"]) /
            (out[f"{spec.prefix}_roll5_std"].abs() + 1e-6)
        ).astype(np.float32)

    return out


# ============================================================
# Encoding
# ============================================================
def fit_category_encoders(train_df: pd.DataFrame, test_df: pd.DataFrame) -> Dict[str, Dict[str, int]]:
    encoders: Dict[str, Dict[str, int]] = {}
    for col in ["code", "sub_code", "sub_category"]:
        all_values = pd.concat([train_df[col].astype(str), test_df[col].astype(str)], axis=0).unique().tolist()
        encoders[col] = {v: i for i, v in enumerate(all_values)}
    return encoders


def apply_category_encoders(df: pd.DataFrame, encoders: Dict[str, Dict[str, int]]) -> pd.DataFrame:
    out = df.copy()
    for col in ["code", "sub_code", "sub_category"]:
        out[f"{col}_enc"] = out[col].astype(str).map(encoders[col]).fillna(-1).astype(np.int32)
    return out


# ============================================================
# Splits
# ============================================================
def make_time_folds(ts: pd.Series, n_folds: int = 4) -> List[Tuple[np.ndarray, np.ndarray]]:
    unique_ts = np.array(sorted(ts.unique()))
    n_unique = len(unique_ts)
    fold_edges = np.linspace(0, n_unique, n_folds + 2, dtype=int)
    folds: List[Tuple[np.ndarray, np.ndarray]] = []

    for i in range(n_folds):
        train_end = fold_edges[i + 1]
        valid_end = fold_edges[i + 2]
        train_ts = set(unique_ts[:train_end])
        valid_ts = set(unique_ts[train_end:valid_end])

        train_idx = np.where(ts.isin(train_ts).values)[0]
        valid_idx = np.where(ts.isin(valid_ts).values)[0]
        if len(train_idx) == 0 or len(valid_idx) == 0:
            continue
        folds.append((train_idx, valid_idx))
    return folds


# ============================================================
# Weights
# ============================================================
def recency_weight(ts_values: np.ndarray, max_ts: int, half_life: float = 800.0) -> np.ndarray:
    delta = max_ts - ts_values
    return np.exp(-np.log(2.0) * delta / half_life).astype(np.float32)


# ============================================================
# Model training
# ============================================================
def get_lgb_params() -> Dict:
    return {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "learning_rate": 0.03,
        "num_leaves": 255,
        "max_depth": -1,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "min_data_in_leaf": 100,
        "lambda_l1": 0.5,
        "lambda_l2": 1.0,
        "seed": RANDOM_STATE,
        "feature_fraction_seed": RANDOM_STATE,
        "bagging_seed": RANDOM_STATE,
        "verbosity": -1,
        "n_jobs": -1,
    }


def fill_na_from_train_medians(train_x: pd.DataFrame, valid_x: pd.DataFrame, test_x: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    med = train_x.median(numeric_only=True)
    return train_x.fillna(med), valid_x.fillna(med), test_x.fillna(med)


def optimize_blend_two_models(y_true: np.ndarray, pred_a: np.ndarray, pred_b: np.ndarray, w: np.ndarray) -> float:
    best_alpha = 0.5
    best_score = -1.0
    for alpha in np.linspace(0.0, 1.0, 21):
        blend = alpha * pred_a + (1.0 - alpha) * pred_b
        score = weighted_rmse_score(y_true, blend, w)
        if score > best_score:
            best_score = score
            best_alpha = float(alpha)
    return best_alpha


# ============================================================
# Optional CatBoost import
# ============================================================
try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False


def train_catboost_single_fold(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    w_train: np.ndarray,
    w_valid: np.ndarray,
    cat_features: List[int],
):
    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        learning_rate=0.03,
        depth=8,
        iterations=3000,
        random_seed=RANDOM_STATE,
        verbose=False,
    )
    model.fit(
        X_train,
        y_train,
        sample_weight=w_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        cat_features=cat_features,
        verbose=False,
    )
    return model


# ============================================================
# Main
# ============================================================
def main():
    print("Loading data...")
    train = pd.read_parquet(TRAIN_PATH)
    # chỉ giữ horizon cần train
    train = train[train["horizon"] == 1].reset_index(drop=True)
    test = pd.read_parquet(TEST_PATH)

    if TARGET_COL not in train.columns:
        raise ValueError(f"Target column '{TARGET_COL}' not found in train data.")

    if WEIGHT_COL not in train.columns:
        raise ValueError(f"Weight column '{WEIGHT_COL}' not found in train data.")

    feature_cols = get_feature_cols(train)
    if len(feature_cols) == 0:
        raise ValueError("No feature_* columns found.")

    print("Basic feature engineering...")
    max_ts = int(train[TS_COL].max())
    train = add_rowwise_meta_features(train, feature_cols)
    test = add_rowwise_meta_features(test, feature_cols)
    train = add_basic_time_features(train, max_ts=max_ts)
    test = add_basic_time_features(test, max_ts=max_ts)

    print("History features for train...")
    train = add_train_history_features(train)

    print("History cache for test...")
    history_cache = build_group_history_cache(train)
    test = add_test_history_features(test, history_cache)

    print("Encoding categories...")
    encoders = fit_category_encoders(train, test)
    train = apply_category_encoders(train, encoders)
    test = apply_category_encoders(test, encoders)

    train = reduce_memory(train)
    test = reduce_memory(test)

    exclude_cols = {
        ID_COL,
        TARGET_COL,
        WEIGHT_COL,
        "code",
        "sub_code",
        "sub_category",
    }
    model_features = [c for c in train.columns if c not in exclude_cols]

    print(f"Total model features: {len(model_features)}")

    oof_pred = np.zeros(len(train), dtype=np.float32)
    test_pred_lgb = np.zeros(len(test), dtype=np.float32)
    test_pred_cat = np.zeros(len(test), dtype=np.float32)
    has_cat = HAS_CATBOOST

    horizon_oof_lgb = []
    horizon_oof_cat = []
    horizon_y = []
    horizon_w = []

    for horizon in HORIZON_VALUES:
        print(f"\n===== Horizon {horizon} =====")
        tr_h = train[train["horizon"] == horizon].copy().reset_index(drop=True)
        te_h = test[test["horizon"] == horizon].copy().reset_index(drop=True)

        if len(tr_h) == 0 or len(te_h) == 0:
            print(f"Skip horizon {horizon} because train/test rows are empty.")
            continue

        folds = make_time_folds(tr_h[TS_COL], n_folds=N_FOLDS)
        if len(folds) == 0:
            print(f"Skip horizon {horizon} because no valid folds were created.")
            continue

        y = tr_h[TARGET_COL].values.astype(np.float32)
        base_w = tr_h[WEIGHT_COL].values.astype(np.float32)
        if USE_RECENCY_WEIGHT:
            rec_w = recency_weight(tr_h[TS_COL].values.astype(np.int32), int(tr_h[TS_COL].max()), RECENCY_HALFLIFE)
            fold_w_all = base_w * rec_w
        else:
            fold_w_all = base_w

        oof_lgb_h = np.zeros(len(tr_h), dtype=np.float32)
        oof_cat_h = np.zeros(len(tr_h), dtype=np.float32)
        test_lgb_h = np.zeros(len(te_h), dtype=np.float32)
        test_cat_h = np.zeros(len(te_h), dtype=np.float32)
        best_iters = []

        for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
            print(f"H{horizon} Fold {fold_id}/{len(folds)}")

            X_tr = tr_h.iloc[tr_idx][model_features].copy()
            y_tr = y[tr_idx]
            w_tr = fold_w_all[tr_idx]

            X_va = tr_h.iloc[va_idx][model_features].copy()
            y_va = y[va_idx]
            w_va = base_w[va_idx]

            X_te = te_h[model_features].copy()

            X_tr, X_va, X_te = fill_na_from_train_medians(X_tr, X_va, X_te)

            lgb_train = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
            lgb_valid = lgb.Dataset(X_va, label=y_va, weight=w_va)

            model = lgb.train(
                get_lgb_params(),
                lgb_train,
                num_boost_round=5000,
                valid_sets=[lgb_train, lgb_valid],
                valid_names=["train", "valid"],
                callbacks=[
                    lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                    lgb.log_evaluation(VERBOSE_EVAL),
                ],
            )

            best_iters.append(model.best_iteration)
            pred_va = model.predict(X_va, num_iteration=model.best_iteration).astype(np.float32)
            pred_te = model.predict(X_te, num_iteration=model.best_iteration).astype(np.float32)

            oof_lgb_h[va_idx] = pred_va
            test_lgb_h += pred_te / len(folds)

            if has_cat:
                cat_features = [X_tr.columns.get_loc(c) for c in ["code_enc", "sub_code_enc", "sub_category_enc", "horizon"] if c in X_tr.columns]
                cat_model = train_catboost_single_fold(
                    X_tr,
                    y_tr,
                    X_va,
                    y_va,
                    w_tr,
                    w_va,
                    cat_features,
                )
                cat_va = cat_model.predict(X_va).astype(np.float32)
                cat_te = cat_model.predict(X_te).astype(np.float32)
                oof_cat_h[va_idx] = cat_va
                test_cat_h += cat_te / len(folds)

            del X_tr, X_va, X_te, lgb_train, lgb_valid, model
            gc.collect()

        score_lgb = weighted_rmse_score(y, oof_lgb_h, base_w)
        print(f"H{horizon} LightGBM OOF score: {score_lgb:.6f}")

        if has_cat:
            score_cat = weighted_rmse_score(y, oof_cat_h, base_w)
            alpha = optimize_blend_two_models(y, oof_lgb_h, oof_cat_h, base_w)
            blend_oof = alpha * oof_lgb_h + (1.0 - alpha) * oof_cat_h
            blend_score = weighted_rmse_score(y, blend_oof, base_w)
            print(f"H{horizon} CatBoost OOF score: {score_cat:.6f}")
            print(f"H{horizon} Blend alpha (LGB weight): {alpha:.2f}")
            print(f"H{horizon} Blend OOF score: {blend_score:.6f}")
            final_test_h = alpha * test_lgb_h + (1.0 - alpha) * test_cat_h
        else:
            alpha = 1.0
            final_test_h = test_lgb_h

        tr_idx_global = train.index[train["horizon"] == horizon]
        te_idx_global = test.index[test["horizon"] == horizon]
        oof_pred[tr_idx_global] = alpha * oof_lgb_h + (1.0 - alpha) * oof_cat_h if has_cat else oof_lgb_h
        test_pred_lgb[te_idx_global] = final_test_h

        horizon_oof_lgb.append(oof_lgb_h)
        if has_cat:
            horizon_oof_cat.append(oof_cat_h)
        horizon_y.append(y)
        horizon_w.append(base_w)

        # Optional final refit on full horizon data
        full_X = tr_h[model_features].copy()
        full_test_X = te_h[model_features].copy()
        med_full = full_X.median(numeric_only=True)
        full_X = full_X.fillna(med_full)
        full_test_X = full_test_X.fillna(med_full)

        if USE_RECENCY_WEIGHT:
            full_sw = base_w * recency_weight(tr_h[TS_COL].values.astype(np.int32), int(tr_h[TS_COL].max()), RECENCY_HALFLIFE)
        else:
            full_sw = base_w

        best_iter = max(200, int(np.mean(best_iters) * 1.05)) if best_iters else 1500
        final_model = lgb.LGBMRegressor(**{k: v for k, v in get_lgb_params().items() if k not in ["metric", "verbosity", "n_jobs"]}, n_estimators=best_iter)
        final_model.fit(full_X, y, sample_weight=full_sw)
        final_pred_full = final_model.predict(full_test_X).astype(np.float32)
        test_pred_lgb[te_idx_global] = 0.5 * test_pred_lgb[te_idx_global] + 0.5 * final_pred_full

        del tr_h, te_h, full_X, full_test_X, final_model
        gc.collect()

    overall_score = weighted_rmse_score(
        train[TARGET_COL].values.astype(np.float32),
        oof_pred,
        train[WEIGHT_COL].values.astype(np.float32),
    )
    print(f"\nOverall OOF score: {overall_score:.6f}")

    submission = pd.DataFrame({ID_COL: test[ID_COL], "prediction": test_pred_lgb.astype(np.float32)})
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved submission to {SUBMISSION_PATH}")


if __name__ == "__main__":
    main()

Loading data...
Basic feature engineering...
History features for train...
History cache for test...
Encoding categories...
Total model features: 171

===== Horizon 1 =====
H1 Fold 1/4
[200]	train's rmse: 0.000849011	valid's rmse: 0.00112381
H1 Fold 2/4
[200]	train's rmse: 0.000915471	valid's rmse: 0.000950297
H1 Fold 3/4
[200]	train's rmse: 0.000905278	valid's rmse: 0.000847168
H1 Fold 4/4
[200]	train's rmse: 0.000800004	valid's rmse: 0.00111644
H1 LightGBM OOF score: 0.034928
H1 CatBoost OOF score: 0.025532
H1 Blend alpha (LGB weight): 0.60
H1 Blend OOF score: 0.040858

===== Horizon 3 =====
Skip horizon 3 because train/test rows are empty.

===== Horizon 10 =====
Skip horizon 10 because train/test rows are empty.

===== Horizon 25 =====
Skip horizon 25 because train/test rows are empty.

Overall OOF score: 0.000000
Saved submission to /kaggle/working/submission.csv
